# Projeto Final: Data Driven Models & ML

Esse notebook tem como objetivo gerar os resultados vistos no relatório crowd_counting_report. 

Bibliotecas:

In [ ]:
### Necessary Libraries: numpy pandas matplotlib plotly torch torchvision torchaudio scikit-learn tensorboard tqdm kagglehub pillow ipython

# =========================
# Basic
# =========================
import copy
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

# =========================
# Plotting
# =========================
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.functional import cross_entropy
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from torchvision.models.vision_transformer import VisionTransformer as ViT

# =========================
# Scikit-learn
# =========================
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold, ShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

# =========================
# TensorBoard
# =========================
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# =========================
# Images
# =========================
from IPython.display import Video
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

# =========================
# Other
# =========================
import kagglehub
import tqdm

## Downloading the Dataset

**Kaggle hub:** The dataset beign used is the kaggle Crowd-UIT dataset by user khitthanhnguynphan. IT can be downloaded by importing kagglehub and setting the two environment avriables below to both your kaggle username and kaggle api key, both of each that are easilly avaliable on your kaggle website profile.

In [ ]:

os.environ["KAGGLE_USERNAME"] = ""
os.environ["KAGGLE_KEY"] = r""

# exammple
# os.environ["KAGGLE_USERNAME"] = "vitormartinsgouvea"
# os.environ["KAGGLE_KEY"] = r"12345_abc"


**Reproducibilidade:** Various seeds for numpy, pytorch and random as well as plotting styles for matplotlib as onther plotting libraries. torch_deterministic_algorithms simply changes pytorch so that everytime it has to do an operation where there is a deterministic way and a random(possibly faster) way of  calcualting something, it chooses the deterministic in order to make the code more reproducable.

In [ ]:

seed = 22604
np.random.seed(seed)
plt.style.use('ggplot')
pio.renderers.default = 'notebook'

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


# Make PyTorch deterministic
torch.use_deterministic_algorithms(True)

g = torch.Generator()
g.manual_seed(seed)


Detects and sets the device for all pytorch operations. Since i dont have acces to a nivdia gpu and the process to use my intel one seemed too complicated, i simply ran the code with my cpu. Perhaps there are certain bugs that arise when using a gpu to run the code that i dont know about, altough i checked with device="cpu" and it seems to work fine.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

Downloads the dataset using kaggle hub and saves it´s path as a varible. If the dataset has been downloaded on another way, define the path variable manually as in example.

In [ ]:
# Download latest version from kaggle hub
path = kagglehub.dataset_download("khitthanhnguynphan/crowduit")

# Example: Manually setup path
# path = rf"users\..."

print("Path to dataset files:", path)

## Data preprocessing

### Visualzando os Dados & Reformatando

Visualizing the videos

In [ ]:
dataset_path = path + r"\Crowd-UIT"

video_id = 1 # number of the video, goes from 1 to 10
video_path = dataset_path + rf"\Video\{video_id}.mp4"

Video(filename=video_path, embed=True, width=720, height=480)

**Importing and reformatting the dataset**: The original dataset is composed of 3 files: one is called *Video* and contains the actual 10 videos themselves; the other is called *Image* and contains 10 folders, each with the frames of the respective video; the final one is called *Json* and contains 10 folders, each one with the repsective counts for each frame in Image. In addition to the count o poeple on frame, it also gives their position, as well as the frame id and video id.

The following code combines these three folders into one dictionary (frames) for ease of manipulation later on. The dictionary stores each sample as the information contained on the Json alongside a path to the image that Json references. 

In [ ]:

# ---------------------------------------------------------------------
# Dataset root
# ---------------------------------------------------------------------
ROOT = Path(dataset_path)

images_root = ROOT / "Image"
json_root = ROOT / "Json"

# ---------------------------------------------------------------------
# Dictionary indexed by video number
# ---------------------------------------------------------------------
videos = {}

for video_folder in sorted(images_root.iterdir(), key=lambda p: int(p.name)):

    if not video_folder.is_dir():
        continue

    video_id = int(video_folder.name)
    annotation_folder = json_root / video_folder.name

    frames = []

    image_files = sorted(
        video_folder.glob("*"),
        key=lambda p: int(p.stem)
    )

    for image_path in image_files:

        frame_number = int(image_path.stem)

        json_path = annotation_folder / f"{frame_number}.json"

        if not json_path.exists():
            raise FileNotFoundError(
                f"Missing annotation:\n{json_path}"
            )

        with open(json_path, "r") as f:
            people = json.load(f)

        frames.append({
            "video_id": video_id,
            "frame": frame_number,
            "image_path": image_path,
            "people": people,
            "count": len(people)
        })

    videos[video_id] = frames

In [ ]:
total_frames = np.sum([len(videos[i]) for i in range(1, len(videos)+1)])

print(f"Total number of frames (i.e. samples) is {total_frames}")

Visualizing all scenes

In [ ]:
frame_id = 10

image_list = [Image.open(videos[video_id][frame_id]['image_path']) for video_id in range(1,11)] # all frames with that id across all videos


fig, axes = plt.subplots(3, 3, figsize=(16,16))

axes = axes.ravel()

for ax, (video_id, image) in zip(axes, enumerate(image_list)):

    ax.imshow(image, cmap="gray" if image.mode=="L" else None)
    ax.set_title(f"Video {video_id+1}", fontsize=10)
    ax.axis("off")

# Hide unused axes
for ax in axes[len(image_list):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

As we can see the total number of samples in our dataset is relativally low, with only 1095 samples.
In order to improve performance and generalization, we augment the dataset with some transformations. A few candidate transformations can be visualized below:

### **Data augmentation:**
We visualize all torchvision/Pillow image transformations available and select which ones seem more appropriatte and will help the model generalize. Dully note that since these transofrmations are in both Pillow and Torchvision, it is possible to fully replace the transformations below with torchvision native transformations if one wishes to not install Pillow.

In [ ]:
def visualize_transforms(img):

    transforms_dict = {

        "Original":
            img,

        "Brightness +50%":
            ImageEnhance.Brightness(img).enhance(1.5), 

        "Brightness -50%":
            ImageEnhance.Brightness(img).enhance(0.5),

        "Contrast +50%":
            ImageEnhance.Contrast(img).enhance(1.5),

        "Contrast -50%":
            ImageEnhance.Contrast(img).enhance(0.5),

        "Saturation +50%":
            ImageEnhance.Color(img).enhance(1.5),

        "Saturation -50%":
            ImageEnhance.Color(img).enhance(0.5),

        "Horizontal Flip":
            ImageOps.mirror(img),

        "Gaussian Blur":
            img.filter(ImageFilter.GaussianBlur(radius=2)),

        "Grayscale":
            ImageOps.grayscale(img),

        "Invert":
            ImageOps.invert(img),

        "Color Jitter":
            transforms.ColorJitter(
                brightness=0.3,
                contrast=0.3,
                saturation=0.3,
                hue=0.1
            )(img),

        "Rotation (+15°)":
            img.rotate(15),

        "Rotation (-15°)":
            img.rotate(-15),

        "Sharpness +100%":
            ImageEnhance.Sharpness(img).enhance(2.0),

        "Sharpness -80%":
            ImageEnhance.Sharpness(img).enhance(0.2),
    }

    fig, axes = plt.subplots(4, 4, figsize=(16,16))

    axes = axes.ravel()

    for ax, (name, image) in zip(axes, transforms_dict.items()):

        ax.imshow(image, cmap="gray" if image.mode=="L" else None)
        ax.set_title(name, fontsize=10)
        ax.axis("off")

    # Hide unused axes
    for ax in axes[len(transforms_dict):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

transform comparisons

In [ ]:
video_id = 5 # video where the frame comes from
frame_id = 90 # video frame

img_path = videos[video_id][frame_id]['image_path'] # gets the path variable for that especific image
img = Image.open(img_path)
visualize_transforms(img)

Since in most applications the data beign processed by the model will be from cameras or some other form of fixed POV, **rotation\perspective** transforms are not going to be used. 
Given our objetive, **grayscale** and **color inversion** were also discarded since they takes away the color information that can be useful for indentifing people that are partially hidden or in darker places of the scene. Since many images dont visually appear to change when increasing or decreasing **sharpness**, it was also excluded. 

The remaining transforms are all going to be randomly applied during training in an online fashion, meaning each time a batch is selected the neural network will see a (possibly) randomly augmented image. This makes the number of possible  different augmentations theoretically infinite and avoids the storage of a large number of different augmentations.   
Note that most transforms have an interpretation in regard to scene conditions: **Brightness**, **Constrast** and **Saturation** can all be interpreted as weather conditions(e.g. cloudy, sunny), while the **Gaussian blur** can be interpreted as the video quality of the camera.

**Random Transform:** The traditional image augmentation pipeline (that i saw in my previous machine learning course anyway) would have each sample be augmented with a single transformation or a sequence of transformations and stored alongside the original image in order to increase the dataset size. 

Despite beign easier to keep track of, several issues arise with this approach, such as: which transforms to apply, which parameters to use in each transform, should they be applied each one time of multiple times to the same image and how do we store the amimum number of augmentations without using too much memory. 

A solution is to do augmentation *online*, that is: instead of augmenting each image before the whole train-test splitting, the augmentation is done randomly each time the dataloader loads a batch of iamges. That way, in the same batch we can have an image that hasn´t been altered at all with an image that has been altered with all transformations available. In order to ensure that each batch is as vaired as possible, the parameters of each trasnformation are also choosen randomly from a range, with the transformation itself having a probability of taking place. 
 


In [ ]:
aug_prob = 0.75 # number of images that are augmented (on average)
transform_prob = 0.5 / aug_prob # number of images that will receive each transform (on average)
# writing transform_prob this way makes it so we can control the total fraction of images that receive each transform


train_augmentation = transforms.Compose([

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.2, # picks a value uniformally from 80% to 120% to be the brightness val 
            contrast=0.2, # same as brightness
            saturation=0.2, # same as brightness
            hue=0.05 # same as brightness
        )
    ], p=transform_prob), # probability that the transform will take place 

    transforms.RandomHorizontalFlip(transform_prob), # probability that the transform will take place 

    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3, # 3x3 gaussian kernel 
            sigma=(0.1,1.0) # controls blur intensity
        )
    ], p=transform_prob), # probability that the transform will take place 

])

## Splitting the Data

As mentioned we will hold out a video to use as a testing dataset. We chose **video 6** (visualized below) for the following reasons: 
- The distance/perspective from the camera to the crowd is similar to video 8 and not too different from other videos, which means that test performance is most likely not going to be influenced by such factors. If we instead used video 5 for example, it would be reasonable to blame poor test performance on the large camera distance from the crowd, distance that was much smaller in the train dataset.

- Another argument is that the scene appears to take place on a tourist location. This is a type of place that can benefit from a crowd counting model since such places dont always have a quantitative measure of how many people are at a site, but nevertheless need to control the number of people to avoid discomfort and possible injury caused by overcrowding.  

In [ ]:

video_path = dataset_path + rf"\Video\{6}.mp4"

Video(filename=video_path, embed=True, width=720, height=480)

### Train and Test

In [ ]:
test_frames = videos[6]

train_videos = {k: v for k, v in videos.items() if k != 6}


Before proceeding, we explicit two problems that can compromise training as well as performance evaluation.

**Firstly**: since the time dependency between frames is not going to be incorporated into the model, there is no need to preserve the sequential structure of the videos and failing to remove such dependency may mislead the model into learning the wrong thing. Luckly a simple shuffle=True somewhere in the training process will fix that.

**Secondly**: because of the videos frame-rates, frames that are next to each other temporally will be very similar, with the crowd moving minimally from one frame to the other. This means that doing a simple K-fold will introduce a form of data-leakage into the validation set. In order to fix that we force videos to be entiraly inside the train or test dataset ny doing groupKfold.

**ImageCache class:** Helps load the data into a form that is ready to be processed, that is, it loads into RAM and resizes the image. Also contains a function that normalizes a given batch of samples in order to normalize the variables using the train data. This separation of duties from the Dataset class will help later on.

In [ ]:
img_size = 224

class ImageCache:

    def __init__(self, samples):

        self.samples = samples

        resize = transforms.Resize((img_size, img_size)) # transforms into square image of size img_size

        self.images = []

        # Cache for normalization statistics
        self._stats_cache = {}

        print(f"Loading {len(samples)} images into RAM...")

        for i, sample in enumerate(samples): # loads and resizes all images in samples and stores them in RAM

            image = (
                Image.open(sample["image_path"])
                .convert("RGB")
            )

            image = resize(image) # resized image

            # Detach from file
            self.images.append(image.copy())

            if (i + 1) % 100 == 0:
                print(f"{i+1}/{len(samples)}")

        print("Done.")

    # -------------------------------------------------------
    # Compute channel-wise mean/std on a subset of images
    # -------------------------------------------------------

    def compute_mean_std(self, indices):

        # Use the indices as the cache key
        key = tuple(indices)

        if key in self._stats_cache: # avoids calculating the mean and std for the same fold everytime a new hyperapramter is tested
            return self._stats_cache[key]

        to_tensor = transforms.ToTensor() # transforms the iamge to tensor for ease of calculation

        channel_sum = torch.zeros(3)
        channel_sq_sum = torch.zeros(3)
        n_pixels = 0

        for idx in indices:

            x = to_tensor(self.images[idx])

            channel_sum += x.sum(dim=(1, 2)) 
            channel_sq_sum += (x ** 2).sum(dim=(1, 2))

            n_pixels += x.shape[1] * x.shape[2]

        mean = channel_sum / n_pixels # mean of each color channel

        std = torch.sqrt(  # std of each color channel
            channel_sq_sum / n_pixels - mean ** 2
        )

        stats = (mean.tolist(), std.tolist())

        # Save for future calls
        self._stats_cache[key] = stats

        return stats

### **Train and Val:**
As mentioned before, we do Group K-Fold Cross Validation, that is, we separate the train dataset into train and validation without breaking groups(in this case videos) up, therefore making it so that both thte train and validation data are actually a list of videos separated in order to guarantee a size balance between the data.

In [ ]:

k = 5 # num of folds

samples = []

for video_id in train_videos.keys():
    samples.extend(train_videos[video_id])

print(f"Total samples: {len(samples)}")


cache = ImageCache(samples) # loads and resizes the train images


groups = np.array([sample["video_id"] for sample in samples])

gkf = GroupKFold(n_splits=k)

folds = []
fold_statistics = {}


for fold_id, (train_idx, val_idx) in enumerate(
    gkf.split(samples, groups=groups),
    start=1
):

    train_videos = sorted(
        np.unique(groups[train_idx]).tolist()
    )

    val_videos = sorted(
        np.unique(groups[val_idx]).tolist()
    )

    train_counts = [
        samples[i]["count"]
        for i in train_idx
    ]

    val_counts = [
        samples[i]["count"]
        for i in val_idx
    ]

    folds.append({
        "train_idx": train_idx,
        "validation_idx": val_idx,
        "train_videos": train_videos,
        "validation_videos": val_videos,
    })

    fold_statistics[fold_id] = {
        "train_counts": train_counts,
        "validation_counts": val_counts,
    }

    print(f"Fold {fold_id}")
    print(f"  Train videos      : {train_videos}")
    print(f"  Validation videos : {val_videos}")
    print(f"  Train samples     : {len(train_idx)}")
    print(f"  Validation samples: {len(val_idx)}")
    print("-" * 60)

**normalized histograms of crowd counts in train and validation:** viszualizes the distribution shift between the train and validation splits. altought to be expected, having a notion of how different the count distribution is between datasets will help make sense of the results.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(8, 15))

for fold_id, ax in enumerate(axes, start=1):

    ax.hist(
        fold_statistics[fold_id]["train_counts"],
        bins=30,
        alpha=0.5,
        label="Train",
        density=True,
    )

    ax.hist(
        fold_statistics[fold_id]["validation_counts"],
        bins=30,
        alpha=0.5,
        label="Validation",
        density=True,
    )

    ax.set_title(f"Fold {fold_id}")
    ax.legend()

plt.tight_layout()

## Training the Model

### Helper Functions & Definitions

**Defining the Dataset Class:** loads the data given indices and a cache and applies a tranform to the image before returning the count and the image. will be used to define the train and validation datasets durign training.

In [ ]:

class CrowdDataset(Dataset):

    def __init__(self, cache, indices, transform=None):

        self.cache = cache
        self.indices = indices

        if transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor()
            ])
        else:
            self.transform = transform

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, idx):

        sample_idx = self.indices[idx]

        sample = self.cache.samples[sample_idx]

        image = self.cache.images[sample_idx].copy()

        image = self.transform(image)
        #print(type(image))

        return {
            "image": image,
            "count": torch.tensor(
                sample["count"],
                dtype=torch.float32
            ),
            "video_id": sample["video_id"],
            "frame": sample["frame"],
        }

Model Training Function

In [ ]:



def train_model(
    model, # model that is going to be trained
    train_loader, # loader function that loads data from a Dataset for the model to process during training
    val_loader, # loader function that loads data from a Dataset for the model to process during validation
    optimizer, # optmizer 
    criterion, # loss function
    n_epochs, # maximum number of epochs, i.e. passes trough the full dataset
    device, # cuda or cpu
    writer=None, # function that sends data to tensorboard, must depend on the fold and hyperapramter in order to separate different stats
    scheduler=None, # changes the learning rate dynamically along the training process in order to better explore the loss landscape
    patience=5, # maximum number of epochs withou any significant change in the loss function before the training loop breaks 
    checkpoint_path=None, # path to register general training statistics, avois losing the enitre training progress because of hardware or somthing like that
):

    history = {
        "train_loss": [],
        "val_loss": [],
        "lr": []
    }

    model = model.to(device)

    # initialize variables
    best_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    for epoch in range(n_epochs):
        
        ###########################################################
        # Training

        model.train() # puts model in training mode, that is, makes certain parts of the architecture that only are active in training work, such as droput or batchnorm

        running_loss = 0.0
        n_samples = 0

        epoch_start = time.perf_counter()

        last_time = epoch_start

        for i, batch in enumerate(train_loader):

            # ----------------------------
            # Loadin the data
            load_time = time.perf_counter() - last_time # time that it takes to load the batch, helpfull for debuggin

            images = batch["image"].to(device, non_blocking=True) # bacth images
            targets = batch["count"].float().to(device, non_blocking=True) # batch counts
            
            # -------------------
            # Forward + backward
            t0 = time.perf_counter()

            optimizer.zero_grad() # sets gradients to 0 in order to prevent gradients from previous batches from accumulating 

            predictions = model(images).squeeze(-1) 

            loss = criterion(predictions, targets)

            global_step = epoch * len(train_loader) + i # used in plotting training loss vs batch
            if writer is not None:
    
                writer.add_scalar(
                    "Loss/Train_batch",
                    loss.item(),
                    global_step
                )

            loss.backward() # backprop

            total_norm = 0 # sets grad norm to 0 to prevent gradietn norms from previous batches from accumulating

            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.norm(2).item() ** 2

            total_norm = total_norm ** 0.5

            if writer is not None:
                writer.add_scalar(
                    "Gradient/Norm",
                    total_norm,
                    global_step,
                )
                
            forward_backward_time = time.perf_counter() - t0 # time spent on the forward and backward passes

            # --------------
            # Optimizer step
            t0 = time.perf_counter()

            optimizer.step() 
            
            if scheduler is not None:
                scheduler.step() # updates the learning rate           

            optimizer_time = time.perf_counter() - t0

            # -------------------
            # Gradient histograms
            if writer is not None:

                for name, param in model.named_parameters():

                    if param.grad is not None:

                        writer.add_histogram(
                            f"{name}.grad",
                            param.grad.detach().cpu(),
                            epoch,
                        )

            # ----------
            # Statistics
            batch_size = images.size(0)

            running_loss += loss.item() * batch_size
            n_samples += batch_size

            # ------------------------
            # Progress every 5 batches
            if i % 5 == 0:

                print(
                    f"Batch {i+1:2d}/{len(train_loader)} | "
                    f"Load: {load_time:.3f}s | "
                    f"Forward+Backward: {forward_backward_time:.3f}s | "
                    f"Optimizer: {optimizer_time:.3f}s"
                )

            last_time = time.perf_counter()

        train_loss = running_loss / n_samples
        history["train_loss"].append(train_loss)
        history["lr"].append(optimizer.param_groups[0]["lr"])

        print(f"\nEpoch training time: {time.perf_counter() - epoch_start:.2f}s")


        if val_loader is not None:
            ###########################################################
            # Validation
        
            model.eval() # turns off things like batchnorm and dropout

            running_loss = 0.0
            n_samples = 0

            with torch.no_grad(): # stops the gradient from beign updated
                

                for batch in val_loader:

                    images = batch["image"].to(device, non_blocking=True)
                    targets = batch["count"].float().to(device, non_blocking=True)

                    predictions = model(images).squeeze(-1)

                    loss = criterion(predictions, targets)

                    batch_size = images.size(0)

                    running_loss += loss.item() * batch_size
                    n_samples += batch_size

            val_loss = running_loss / n_samples

            # Scheduler
        
            # Save history
        
            history["val_loss"].append(val_loss)

            ###########################################################
            # Early stopping

            if val_loss < best_loss:

                best_loss = val_loss

                best_weights = copy.deepcopy(
                    model.state_dict()
                )

                epochs_without_improvement = 0

                if checkpoint_path is not None:

                    Path(checkpoint_path).parent.mkdir(
                        parents=True,
                        exist_ok=True
                    )

                    torch.save(
                        model.state_dict(),
                        checkpoint_path
                    )   

            else:

                epochs_without_improvement += 1

            # TensorBoard Update
            if writer is not None:
            
                writer.add_scalar(
                    "Loss/Train",
                    train_loss,
                    epoch
                )

                writer.add_scalar(
                    "Loss/Validation",
                    val_loss,
                    epoch
                )

                writer.add_scalar(
                    "Learning Rate",
                    optimizer.param_groups[0]["lr"],
                    epoch
                )

                rmse = np.sqrt(val_loss)
                writer.add_scalar(
                    "Metrics/RMSE",
                    rmse,
                    epoch
                )

                if epoch % 5 == 0:

                    for name, param in model.named_parameters():

                        writer.add_histogram(
                            name,
                            param.detach().cpu(),
                            epoch,
                        )
                    writer.flush()

            ###########################################################
            # Progress
        
            print(
                f"Epoch {epoch+1:3d} | "
                f"Train={train_loss:.4f} | "
                f"Val={val_loss:.4f}"
            )

            ###########################################################
            # Stop?
        
            if epochs_without_improvement >= patience:

                print("Early stopping.")
                break
            # print(os.listdir(writer.log_dir))

            # ea = EventAccumulator("runs/hidden_64/fold_1")
            # ea.Reload()

            # print(ea.Tags())

    ###############################################################
    # Restore best weights
   
    if val_loader is not None:
        model.load_state_dict(best_weights)

    return model, history



Defining model arch

In [ ]:

class CrowdCNN(nn.Module):

    def __init__(
        self,
        base_channels=32,
        num_blocks=4,
        kernel_size=3,
        hidden_dim=128,
        dropout=0.3,
    ):

        super().__init__()

        padding = kernel_size // 2

        layers = []

        in_channels = 3
        out_channels = base_channels

        for _ in range(num_blocks):

            layers.extend([
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    padding=padding,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels), # regulriza em batch a camada, evit aproblemas de vanishing/exploding gradient
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ])

            in_channels = out_channels
            out_channels *= 2

        layers.append(nn.AdaptiveAvgPool2d(1))

        self.features = nn.Sequential(*layers)

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout), # induces variability into the DNN
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):

        x = self.features(x)
        x = self.regressor(x)

        return x

## Hyperparameter Tuning:

### Learning Rate

Finding the best learning rate

In [ ]:

lr_grid = [1e-4,3e-4,1e-3]
n_epochs = 50
patience = 10

cv_results = {}
results = {}

for lr in lr_grid:

    print(f"\n{'='*60}")
    print(f"Training CNN with lr = {lr}")
    print(f"{'='*60}")

    results[lr] = defaultdict(dict)
    cv_results[lr] = {
    "fold_losses": []
}
    

    for fold_id, fold in enumerate(folds, start=1):

        writer = SummaryWriter( 
            log_dir=f"runs/hidden_{lr}/fold_{fold_id}"
        )

        print(f"\nFold {fold_id}/{k}")

        # --------------------------------
        # Compute normalization statistics
        

        mean, std = cache.compute_mean_std(fold["train_idx"])

     

        # ----------
        # Transforms
        
        train_transform = transforms.Compose([
            train_augmentation,
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

        val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    
        # --------
        # Datasets
       

        train_dataset = CrowdDataset(
            cache=cache,
            indices=fold["train_idx"],
            transform=train_transform,
        )

        val_dataset = CrowdDataset(
            cache=cache,
            indices=fold["validation_idx"],
            transform=val_transform,
        )

        # -----------
        # Dataloaders
       

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
            generator=g,
            num_workers=0,
            persistent_workers=False,
            pin_memory=(device.type == "cuda"), # uses pin memory if there is a gpu 
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False,
        )

        # -----
        # Model
        

        model = CrowdCNN(
                base_channels=32,
                num_blocks=4,
                kernel_size=3,
                hidden_dim=64,
                dropout=0.3,
            )
  
        # ----------------------------
        # Optimizer / Loss / Scheduler
        
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=1e-4,
        )

        criterion = nn.L1Loss()
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=lr,
                epochs=n_epochs,
                steps_per_epoch=len(train_loader),
                pct_start=0.3,
            )

        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------

        model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            scheduler=scheduler,
            writer=writer,
            n_epochs=n_epochs,
            patience=10,
            checkpoint_path=f"checkpoints/vit_hidden{lr}_fold{fold_id}.pt",
        )

        best_val = min(history["val_loss"])

        cv_results[lr]["fold_losses"].append(best_val)

        print(f"Best validation loss: {best_val:.4f}")

        example_batch = next(iter(train_loader))

        writer.add_graph(
            model,
            example_batch["image"].to(device)
        )

        writer.flush()

        # --------------------------------------------------------
        # Validation predictions
        # --------------------------------------------------------

        model.eval()

        with torch.no_grad():

            for batch in val_loader:

                images = batch["image"].to(device, non_blocking=True)
                targets = batch["count"].float().to(device, non_blocking=True)

                preds = model(images).squeeze(-1)

                sq_errors = (preds - targets) ** 2

                for video_id, frame, pred, target, err in zip(
                    batch["video_id"].tolist(),
                    batch["frame"].tolist(),
                    preds.cpu().numpy(),
                    targets.cpu().numpy(),
                    sq_errors.cpu().numpy(),
                ):

                    if video_id not in results[lr]:

                        results[lr][video_id] = {
                            "frame": [],
                            "prediction": [],
                            "ground_truth": [],
                            "squared_error": [],
                            "fold": fold_id,
                        }

                    results[lr][video_id]["frame"].append(frame)
                    results[lr][video_id]["prediction"].append(float(pred))
                    results[lr][video_id]["ground_truth"].append(float(target))
                    results[lr][video_id]["squared_error"].append(float(err))

        writer.close()


    mean_val = np.mean(cv_results[lr]["fold_losses"])
    std_val = np.std(cv_results[lr]["fold_losses"])

    cv_results[lr]["mean"] = mean_val
    cv_results[lr]["std"] = std_val

    print()
    print(f"Learning rate {lr}")
    print(f"Mean validation loss = {mean_val:.4f}")
    print(f"Std                 = {std_val:.4f}")
    print()


best_lr = min(
    cv_results,
    key=lambda lr: cv_results[lr]["mean"]
)

print("=" * 60)
print("Cross-validation results")
print("=" * 60)

for lr in lr_grid:
    print(
        f"lr={lr:<8} "
        f"mean={cv_results[lr]['mean']:.4f} "
        f"std={cv_results[lr]['std']:.4f}"
    )

print("\nBest learning rate:", best_lr)

Summary of Cross-Validation results: learning rate

In [ ]:

summary = pd.DataFrame({
    "learning_rate": list(cv_results.keys()),
    "mean_val_loss": [cv_results[k]["mean"] for k in cv_results],
    "std_val_loss": [cv_results[k]["std"] for k in cv_results],
})

summary = summary.sort_values("mean_val_loss")

summary.to_csv("cv_summary.csv", index=False)

Validadtion Loss vs Hyperparameters Plot: learning rate

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = summary["learning_rate"]
y = summary["mean_val_loss"]
std = summary["std_val_loss"]

ax.plot(
    x,
    y,
    "-o",
    color="tab:blue",
    linewidth=2,
    markersize=7,
    label="Mean validation loss",
)

ax.fill_between(
    x,
    y - std,
    y + std,
    color="tab:blue",
    alpha=0.2,
    label=r"$\pm1$ std",
)

ax.set_xscale("log")
ax.set_xlabel("Learning Rate")
ax.set_ylabel("Validation Loss")
ax.set_title("Cross-validation Performance")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

### Hidden Dimension

Finding the best hidden_dim:

In [ ]:

opt_learning_rate = 0.001

hidden_dim_grid = [32, 64, 128, 256]
n_epochs = 50
patience = 10

cv_results = {}
results = {}

for hidden_dim in hidden_dim_grid:

    print(f"\n{'='*60}")
    print(f"Training CNN with hidden_dim = {hidden_dim}")
    print(f"{'='*60}")

    results[hidden_dim] = defaultdict(dict)
    cv_results[hidden_dim] = {
    "fold_losses": []
}
    

    
    for fold_id, fold in enumerate(folds, start=1):

        writer = SummaryWriter(
            log_dir=f"runs/hidden_dim_runs/hidden_dim_{hidden_dim}/fold_{fold_id}"
        )

        # print(writer.log_dir)
        print(f"\nFold {fold_id}/{k}")

        # --------------------------------------------------------
        # Compute normalization statistics
        # --------------------------------------------------------

        mean, std = cache.compute_mean_std(fold["train_idx"])

    
        # --------------------------------------------------------
        # Transforms
        # --------------------------------------------------------

        train_transform = transforms.Compose([
            train_augmentation,
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

        val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

    
        
        # --------------------------------------------------------
        # Datasets
        # --------------------------------------------------------

        train_dataset = CrowdDataset(
            cache=cache,
            indices=fold["train_idx"],
            transform=train_transform,
        )

        val_dataset = CrowdDataset(
            cache=cache,
            indices=fold["validation_idx"],
            transform=val_transform,
        )

        # --------------------------------------------------------
        # Dataloaders
        # --------------------------------------------------------

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
            generator=g,
            num_workers=0,
            persistent_workers=False,
            pin_memory=(device.type == "cuda"), # uses pin memory if there is a gpu 
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False,
            pin_memory=(device.type == "cuda"),
        )

        # --------------------------------------------------------
        # Model
        # --------------------------------------------------------

        model = CrowdCNN(
                base_channels=32,
                num_blocks=4,
                kernel_size=3,
                hidden_dim=hidden_dim,
                dropout=0.3,
            )
        

        # --------------------------------------------------------
        # Optimizer / Loss / Scheduler
        # --------------------------------------------------------

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=opt_learning_rate,
            weight_decay=1e-4,
        )

        criterion = nn.L1Loss()
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=opt_learning_rate,
                epochs=n_epochs,
                steps_per_epoch=len(train_loader),
                pct_start=0.3,
            )

        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------

        model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            scheduler=scheduler,
            writer=writer,
            n_epochs=n_epochs,
            patience=10,
            checkpoint_path=f"checkpoints/cnn_hidden_{hidden_dim}_fold{fold_id}.pt",
        )

        best_val = min(history["val_loss"])

        cv_results[hidden_dim]["fold_losses"].append(best_val)

        print(f"Best validation loss: {best_val:.4f}")

        example_batch = next(iter(train_loader))

        writer.add_graph(
            model,
            example_batch["image"].to(device)
        )

        writer.flush()

        # --------------------------------------------------------
        # Validation predictions
        # --------------------------------------------------------

        model.eval()

        with torch.no_grad():

            for batch in val_loader:

                images = batch["image"].to(device, non_blocking=True)
                targets = batch["count"].float().to(device, non_blocking=True)

                preds = model(images).squeeze(-1)

                sq_errors = (preds - targets) ** 2

                for video_id, frame, pred, target, err in zip(
                    batch["video_id"].tolist(),
                    batch["frame"].tolist(),
                    preds.cpu().numpy(),
                    targets.cpu().numpy(),
                    sq_errors.cpu().numpy(),
                ):

                    if video_id not in results[hidden_dim]:

                        results[hidden_dim][video_id] = {
                            "frame": [],
                            "prediction": [],
                            "ground_truth": [],
                            "squared_error": [],
                            "fold": fold_id,
                        }

                    results[hidden_dim][video_id]["frame"].append(frame)
                    results[hidden_dim][video_id]["prediction"].append(float(pred))
                    results[hidden_dim][video_id]["ground_truth"].append(float(target))
                    results[hidden_dim][video_id]["squared_error"].append(float(err))

        writer.close()


    mean_val = np.mean(cv_results[hidden_dim]["fold_losses"])
    std_val = np.std(cv_results[hidden_dim]["fold_losses"])

    cv_results[hidden_dim]["mean"] = mean_val
    cv_results[hidden_dim]["std"] = std_val

    print()
    print(f"Hidden Dimension {hidden_dim}")
    print(f"Mean validation loss = {mean_val:.4f}")
    print(f"Std                 = {std_val:.4f}")
    print()


best_hidden_dim = min(
    cv_results,
    key=lambda hidden_dim: cv_results[hidden_dim]["mean"]
)

print("=" * 60)
print("Cross-validation results")
print("=" * 60)

for hidden_dim in hidden_dim_grid:
    print(
        f"hidden_dim={hidden_dim:<8} "
        f"mean={cv_results[hidden_dim]['mean']:.4f} "
        f"std={cv_results[hidden_dim]['std']:.4f}"
    )

print("\nBest hidden dim:", best_hidden_dim)

summary of cross validation: hidden dim

In [ ]:
summary = pd.DataFrame({
    "hidden_dim": hidden_dim_grid,
    "mean_val_loss": [cv_results[h]["mean"] for h in hidden_dim_grid],
    "std_val_loss": [cv_results[h]["std"] for h in hidden_dim_grid],
})

summary = summary.sort_values("mean_val_loss")

summary.to_csv(
    "hidden_dim_cv_results.csv",
    index=False,
)

val performance vs hyperparameters: hidden_dim

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = summary["hidden_dim"]
y = summary["mean_val_loss"]
std = summary["std_val_loss"]

ax.plot(
    x,
    y,
    "-o",
    color="tab:blue",
    linewidth=2,
    markersize=7,
    label="Mean validation loss",
)

ax.fill_between(
    x,
    y - std,
    y + std,
    color="tab:blue",
    alpha=0.2,
    label=r"$\pm1$ std",
)

#ax.set_xscale("log")
ax.set_xlabel("Hidden Dimension")
ax.set_ylabel("Validation Loss")
ax.set_title("Cross-validation Performance")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

### Base Channel Dimension

Finding the best base channel:

In [ ]:

opt_learning_rate = 0.001
opt_hidden_dim = 256

base_channel_grid =  [16, 32, 64, 128]
n_epochs = 50
patience = 10

cv_results = {}
results = {}

for base_channel in base_channel_grid:

    print(f"\n{'='*60}")
    print(f"Training CNN with base_channel = {base_channel}")
    print(f"{'='*60}")

    results[base_channel] = defaultdict(dict)
    cv_results[base_channel] = {
    "fold_losses": []
}
    

    
    for fold_id, fold in enumerate(folds, start=1):

        writer = SummaryWriter(
            log_dir=f"runs/base_channel_runs/base_channel_{base_channel}/fold_{fold_id}"
        )

        print(f"\nFold {fold_id}/{k}")

        # --------------------------------------------------------
        # Compute normalization statistics
        # --------------------------------------------------------

        mean, std = cache.compute_mean_std(fold["train_idx"])

    
        # --------------------------------------------------------
        # Transforms
        # --------------------------------------------------------

        train_transform = transforms.Compose([
            train_augmentation,
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])

        val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        
        # --------------------------------------------------------
        # Datasets
        # --------------------------------------------------------

        train_dataset = CrowdDataset(
            cache=cache,
            indices=fold["train_idx"],
            transform=train_transform,
        )

        val_dataset = CrowdDataset(
            cache=cache,
            indices=fold["validation_idx"],
            transform=val_transform,
        )

        # --------------------------------------------------------
        # Dataloaders
        # --------------------------------------------------------

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
            generator=g,
            num_workers=0,
            persistent_workers=False,
            pin_memory=(device.type == "cuda"), # uses pin memory if there is a gpu 
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False,
            pin_memory=(device.type == "cuda"),
        )

        # --------------------------------------------------------
        # Model
        # --------------------------------------------------------

        model = CrowdCNN(
                base_channels=base_channel,
                num_blocks=4,
                kernel_size=3,
                hidden_dim=opt_hidden_dim,
                dropout=0.3,
            )
        
    
        # --------------------------------------------------------
        # Optimizer / Loss / Scheduler
        # --------------------------------------------------------

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=opt_learning_rate,
            weight_decay=1e-4,
        )

        criterion = nn.L1Loss()
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer,
                max_lr=opt_learning_rate,
                epochs=n_epochs,
                steps_per_epoch=len(train_loader),
                pct_start=0.3,
            )

        # --------------------------------------------------------
        # Train
        # --------------------------------------------------------

        model, history = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            criterion=criterion,
            scheduler=scheduler,
            writer=writer,
            n_epochs=n_epochs,
            patience=10,
            checkpoint_path=f"checkpoints/cnn_hidden_{base_channel}_fold{fold_id}.pt",
        )

        best_val = min(history["val_loss"])

        cv_results[base_channel]["fold_losses"].append(best_val)

        print(f"Best validation loss: {best_val:.4f}")

        example_batch = next(iter(train_loader))

        writer.add_graph(
            model,
            example_batch["image"].to(device)
        )

        writer.flush()

        # --------------------------------------------------------
        # Validation predictions
        # --------------------------------------------------------

        model.eval()

        with torch.no_grad():

            for batch in val_loader:

                images = batch["image"].to(device, non_blocking=True)
                targets = batch["count"].float().to(device, non_blocking=True)

                preds = model(images).squeeze(-1)

                sq_errors = (preds - targets) ** 2

                for video_id, frame, pred, target, err in zip(
                    batch["video_id"].tolist(),
                    batch["frame"].tolist(),
                    preds.cpu().numpy(),
                    targets.cpu().numpy(),
                    sq_errors.cpu().numpy(),
                ):

                    if video_id not in results[base_channel]:

                        results[base_channel][video_id] = {
                            "frame": [],
                            "prediction": [],
                            "ground_truth": [],
                            "squared_error": [],
                            "fold": fold_id,
                        }

                    results[base_channel][video_id]["frame"].append(frame)
                    results[base_channel][video_id]["prediction"].append(float(pred))
                    results[base_channel][video_id]["ground_truth"].append(float(target))
                    results[base_channel][video_id]["squared_error"].append(float(err))

        writer.close()


    mean_val = np.mean(cv_results[base_channel]["fold_losses"])
    std_val = np.std(cv_results[base_channel]["fold_losses"])

    cv_results[base_channel]["mean"] = mean_val
    cv_results[base_channel]["std"] = std_val

    print()
    print(f"Base Channel {base_channel}")
    print(f"Mean validation loss = {mean_val:.4f}")
    print(f"Std                 = {std_val:.4f}")
    print()


best_base_channel = min(
    cv_results,
    key=lambda base_channel: cv_results[base_channel]["mean"]
)

print("=" * 60)
print("Cross-validation results")
print("=" * 60)

for base_channel in base_channel_grid:
    print(
        f"base_channel={base_channel:<8} "
        f"mean={cv_results[base_channel]['mean']:.4f} "
        f"std={cv_results[base_channel]['std']:.4f}"
    )

print("\nBest base_channel:", best_base_channel)

summary of cross validation: base_channel

In [ ]:
summary = pd.DataFrame({
    "base_channel": base_channel_grid,
    "mean_val_loss": [cv_results[h]["mean"] for h in base_channel_grid],
    "std_val_loss": [cv_results[h]["std"] for h in base_channel_grid],
})

summary = summary.sort_values("mean_val_loss")

summary.to_csv(
    "base_channel_cv_results.csv",
    index=False,
)

val performance vs hyperparameters: base_channel

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = summary["base_channel"]
y = summary["mean_val_loss"]
std = summary["std_val_loss"]

ax.plot(
    x,
    y,
    "-o",
    color="tab:blue",
    linewidth=2,
    markersize=7,
    label="Mean validation loss",
)

ax.fill_between(
    x,
    y - std,
    y + std,
    color="tab:blue",
    alpha=0.2,
    label=r"$\pm1$ std",
)

#ax.set_xscale("log")
ax.set_xlabel("base_channel")
ax.set_ylabel("Validation Loss")
ax.set_title("Cross-validation Performance")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

## Final Model:

In [ ]:
final_lr = 0.001

final_model = CrowdCNN(
                base_channels=best_base_channel, # 64
                num_blocks=4,
                kernel_size=3,
                hidden_dim=best_hidden_dim, # 128
                dropout=0.3,
            )

### final training:

In [ ]:
n_epochs = 50

writer = SummaryWriter(
    log_dir=f"runs/final_runs"
)

print("Final Run")

# --------------------------------
# Compute normalization statistics

mean, std = cache.compute_mean_std(np.arange(len(samples)))

# print(f"Mean: {mean}")
# print(f"Std : {std}")

# ----------
# Transforms

train_transform = transforms.Compose([
    train_augmentation,
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# --------
# Datasets

train_dataset = CrowdDataset(
    cache=cache,
    indices=np.arange(len(samples)),
    transform=train_transform,
)


# -----------
# Dataloaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=g,
    num_workers=0,
    persistent_workers=False,
    pin_memory=(device.type == "cuda"), # uses pin memory if there is a gpu 
)


# -----
# Model

model = final_model

# ----------------------------
# Optimizer / Loss / Scheduler


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=final_lr,
    weight_decay=1e-4,
)

criterion = nn.L1Loss()

scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=final_lr,
        epochs=n_epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.3,
    )

# -----
# Train

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=None,
    optimizer=optimizer,
    device=device,
    criterion=criterion,
    scheduler=scheduler,
    writer=writer,
    n_epochs=n_epochs,
    patience=15,
    checkpoint_path=f"checkpoints/final_train.pt",
)


torch.save(
    {
        "model_state_dict": model.state_dict(),
       "model_hyperparameters": {
        "base_channels": best_base_channel,
        "num_blocks": 4,
        "kernel_size": 3,
        "hidden_dim": best_hidden_dim,
        "dropout": 0.3,
    },
        "optimizer_state_dict": optimizer.state_dict(),
        "mean": mean,
        "std": std,
    },
    "checkpoints/final_model.pt",
)
writer.close()
print("Training finished.")




Tira uma coluna vazia do dataframe(rodar uma vez e descomentar)

In [ ]:
history = {
    k: v
    for k, v in history.items()
    if len(v) > 0
}

pd.DataFrame(history).to_csv(
    "checkpoints/final_history.csv",
    index=False,
)

importando o modelo treinado

In [ ]:
checkpoint = torch.load(
    "checkpoints/final_model.pt",
    map_location=device,
)

trained_model = CrowdCNN(
    **checkpoint["model_hyperparameters"]
).to(device)

trained_model.load_state_dict(checkpoint["model_state_dict"])
trained_model.eval()

mean = checkpoint["mean"]
std = checkpoint["std"]

## Performance de Teste

Definindo conjunto de teste

In [ ]:
test_cache = ImageCache(test_frames)

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std), # mean and std from train data
])

test_dataset = CrowdDataset(
    cache=test_cache,
    indices=np.arange(len(test_frames)),
    transform=test_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
)

### Avaliando no conjunto de teste

In [ ]:
trained_model.eval()

predictions = []
ground_truth = []

with torch.no_grad():

    for batch in test_loader:

        images = batch["image"].to(device, non_blocking=True)
        targets = batch["count"].to(device, non_blocking=True)

        preds = trained_model(images).squeeze(-1)

        predictions.extend(preds.cpu().numpy())
        ground_truth.extend(targets.cpu().numpy())

count vs frame

In [ ]:
fig,ax = plt.subplots()

ax.set_xlabel("frame")
ax.set_ylabel("count")

ax.scatter(range(len(predictions)),predictions,color="lightblue")
ax.plot(predictions,color="lightblue")

ax.scatter(range(len(ground_truth)),ground_truth,color="blue")
ax.plot(ground_truth,color="blue")

test_loss = np.abs(np.array(predictions) - np.array(ground_truth)).sum()
print(f"test loss was= {test_loss}")

### Interpretando resultados

#### Grad-CAM

calcualndo o Grad-CAM

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

sample = test_frames[10]

img = Image.open(sample["image_path"]).convert("RGB")
img = transforms.Resize((224,224))(img)

img_np = np.array(img).astype(np.float32) / 255.0

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

input_tensor = transform(img).unsqueeze(0).to(device)

target_layer = trained_model.features[-5] # ultima layer convolucional

cam = GradCAM(
    model=trained_model,
    target_layers=[target_layer]
)

grayscale_cam = cam(input_tensor=input_tensor)[0]

visualization = show_cam_on_image(
    img_np,
    grayscale_cam,
    use_rgb=True
)


Plotting Grad-CAM

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18,6))

ax[0].imshow(img_np)
ax[0].set_title("Original image")
ax[0].axis("off")

ax[1].imshow(grayscale_cam, cmap="jet")
ax[1].set_title("Grad-CAM")
ax[1].axis("off")

ax[2].imshow(visualization)
ax[2].set_title("Overlay")
ax[2].axis("off")

plt.tight_layout()
plt.show()

#### Visualizing individidual layers

In [ ]:
sample = test_frames[0]

img = Image.open(sample["image_path"]).convert("RGB")
img = transforms.Resize((224, 224))(img)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

x = transform(img).unsqueeze(0).to(device)   # (1,C,H,W)

# -------------------------------------------------------
# Capture feature maps
# -------------------------------------------------------

feature_maps = []

def hook_fn(module, input, output):
    feature_maps.append(output.detach().cpu())

hooks = []

# Register hooks only on Conv2d layers
for layer in trained_model.features:
    if isinstance(layer, torch.nn.Conv2d):
        hooks.append(layer.register_forward_hook(hook_fn))

trained_model.eval()

with torch.no_grad():
    _ = trained_model(x)

for h in hooks:
    h.remove()

# -------------------------------------------------------
# Plot the first few channels from every layer
# -------------------------------------------------------

for layer_idx, fmap in enumerate(feature_maps):

    fmap = fmap.squeeze(0)      # (C,H,W)

    n_show = min(8, fmap.shape[0])

    fig, axes = plt.subplots(
        2, 4,
        figsize=(10,5)
    )

    fig.suptitle(f"Conv Layer {layer_idx+1}")

    for i, ax in enumerate(axes.flat):

        if i < n_show:
            ax.imshow(fmap[i], cmap="viridis")
            ax.set_title(f"Ch {i}")
            ax.axis("off")
        else:
            ax.remove()

    plt.tight_layout()
    plt.show()
